# Introduction to HTRflow (Colab)

This notebook is a Colab-adapted English version of the original notebook.
It clones the official HTRflow repository, switches into the `examples` directory, and then runs the same walkthrough there.


In [ ]:
# Clone the official HTRflow repository and move into the examples directory
%cd /content
!test -d htrflow || git clone https://github.com/AI-Riksarkivet/htrflow.git
%cd /content/htrflow/examples


# Introduction to HTRflow

This is a Jupyter notebook, but it contains very little Python code.
You can also follow it in a regular terminal instead (just remove the `!` in front of the commands).

## What is HTRflow?

HTRflow is the software we use to run HTR projects. At its core it is a Python package, but in practice we use it as a standalone program.
Install it with `pip`:


In [ ]:
!pip install htrflow "transformers<5.0"


We can verify that HTRflow was installed by running `htrflow --help`:


In [ ]:
!htrflow --help


## Transcribing with HTRflow

To transcribe, we use the `htrflow pipeline` subcommand. Let's check which arguments it accepts:


In [ ]:
!htrflow pipeline --help


So we need a *pipeline file* and an image to transcribe. There are some example images in the `images` directory.

*Tip! Their filenames are the National Archives of Sweden image IDs and consist of two parts: a batch ID and a running number. The batch ID is assigned when a volume is scanned. If a volume is scanned more than once, it always gets a new batch ID. To learn more about the images, you can open them in the image viewer using the URL `sok.riksarkivet.se/bildvisning/image-ID`, for example [https://sok.riksarkivet.se/bildvisning/A0068688_00459](https://sok.riksarkivet.se/bildvisning/A0068688_00459).*

We will start by transcribing a small crop from a death and burial record from Sköllersta, outside Örebro:

![](images/C0001358_00076_klipp.png)

To transcribe an image like this, we need to:
1. Segment the lines
2. Put the lines in reading order
3. Transcribe them

There are a few example pipelines in the `pipelines` directory. `pipelines/pipeline1.yaml` is a good fit for this image:


In [ ]:
!cat pipelines/pipeline1.yaml


In [ ]:
!htrflow pipeline pipelines/pipeline1.yaml images/C0001358_00076_klipp.png


If you run this without a GPU, it may take a few minutes per page. When the program is finished, we can inspect the result as plain text:


In [ ]:
!cat outputs/images/C0001358_00076_klipp.txt


Yes — from this transcription we can tell that Anna Olofsdotter, the wife of Jonas Jonsson, has died. She had no children of her own, but was the stepmother of Jonas's five children. She had stomach problems for the last nine years of her life, but was bedridden only during the final eight days before her death.

But the transcription is a bit messy. In particular, there are some inserted short lines between the “main” lines. We also have duplicates (*71. 37.*; her age in years and weeks) and an unclear *C*.

Some of these errors come from the fact that it is simply difficult to represent handwritten text — which really exists in two dimensions — as a single text string. For example, there is a correction on the third line: it says *född i Svennevad 1710 [...]*, but *Svennevad* has been corrected to *Lerbäck*. HTRflow has trouble interpreting this: the original line became *berördnad 1710 [...]* and the correction appears as its own line above it (*terbäck*).

Some of these ambiguities become clearer when we look at the JSON output. There we get extra data for each line, such as where the line is located in the image (polygon and `bbox`, which stands for bounding box) and confidence values from the different models:



In [ ]:
import json

with open("outputs/images/C0001358_00076_klipp.json") as f:
    doc = json.load(f)

doc


The confidence values give us a rough indication of how “certain” the model was about its interpretation. TrOCR usually produces very high confidence values, often 0.9 or higher. You should be careful about reading too much into the absolute number itself (Erik has an ongoing project that uses confidence values to predict CER), but an interpretation with a relatively low confidence score is unlikely to be correct.

If we look at line 17, which consists of a single *C*, we see that it has a confidence value of 0.82, which is fairly low. The confidence value from the segmentation model is 0.3, which is also very low.


In [ ]:
doc["contains"][17]


One way to improve transcription quality is to set thresholds for the confidence values. We could do this as a post-processing step based on the JSON file, but we can also build it into the pipeline so that HTRflow handles it for us. We do that differently depending on the model:

**For YOLO models**, there is built-in support for setting a confidence threshold. We can add that under `generation_settings` in the HTRflow pipeline step. We add this:
```yaml
- step: Segmentation
  settings:
    model: yolo
    model_settings:
      model: Riksarkivet/yolov9-lines-within-regions-1
    generation_settings:    # <--- here we can pass arguments to YOLO
      conf: 0.5   
```
See the documentation for available arguments: [https://docs.ultralytics.com/modes/predict/#inference-arguments](https://docs.ultralytics.com/modes/predict/#inference-arguments).

**For TrOCR**, we cannot apply such a threshold directly, so we use a separate pipeline step:
```yaml
- step: RemoveLowTextConfidenceLines
  settings:
    threshold: 0.85
```
The modified pipeline is available in `pipelines/pipeline2.yaml`.


In [ ]:
!htrflow pipeline pipelines/pipeline2.yaml images/C0001358_00076_klipp.png


Did it get any better? Yes — the strange *C* disappeared. But so did both instances of *71. 37*, which were actually correct. We would have liked to keep at least one of them.


In [ ]:
!cat outputs/images/C0001358_00076_klipp.txt


**It is hard to find confidence thresholds that work well.** Right now, our preference is to let questionable interpretations pass through rather than risk removing correct ones, because that gives us the best possible searchability. But that also makes close reading of a transcribed text more difficult. This is not an obvious priority choice, and there are good arguments on both sides.


## The ALTO and PAGE XML formats

ALTO XML and PAGE XML are two XML-based formats for transcriptions. They are used in applications that display the transcription on top of the image, such as our search service. They contain essentially the same information and impose similar structural requirements on a transcription: there should be regions, within them lines, and preferably words within those lines. The actual text is usually stored on the line or word elements. In our search service we use ALTO XML, while the Transkribus platform primarily uses PAGE XML.

Because both formats require text regions, they work better when we transcribe a full page (rather than just a small crop like above), such as this one: ![page from Svea Court of Appeal](images/A0068688_00459.jpg)

The pipeline `pipelines/pipeline3.yaml` works well for this image. It is roughly the same pipeline as above, but with an additional region segmentation step, and it resembles the pipeline we use when transcribing material for the search service.


In [ ]:
!htrflow pipeline pipelines/pipeline3.yaml images/A0068688_00459.jpg


The result is available in `outputs/images/A0068688_00459.xml`:


In [ ]:
!cat outputs/images/A0068688_00459.xml


## Classification models in or alongside HTRflow

At the moment, there is basic support for running DiT, a classification model based on ViT. The pipeline `pipelines/pipeline4.yaml` is a simple pipeline that runs the image through `microsoft/dit-base-finetuned-rvlcdip`, a checkpoint fine-tuned to classify scanned documents by type. We run the same image through it:



In [ ]:
!htrflow pipeline pipelines/pipeline4.yaml images/A0068688_00459.jpg


In [ ]:
from pprint import pprint
import json

with open("outputs/images/A0068688_00459.json") as f:
    doc = json.load(f)

pprint(doc)


In [ ]:
print("Top 3 classes:")
print(*sorted(doc["classification"].items(), key=lambda item: -item[1])[:3], sep="\n")


There we go — it is handwriting!

However, HTRflow currently cannot act on this classification result. There are definitely cases where we would want to branch based on the output of a classification model:
- Decide whether we should run an image through HTRflow at all. Some images should not be processed with HTR, for example cover pages: [https://sok.riksarkivet.se/bildvisning/C0001358_00001](https://sok.riksarkivet.se/bildvisning/C0001358_00001). When transcribing at scale, it would be valuable to filter such images automatically. Today we do not do that, but we skip all images ending in `00001` to avoid the cover pages.
- Decide which pipeline to use for an image. For example, if the page is printed rather than handwritten, we want to run our OCR model instead of an HTR model. Such a classification might even need to be done at the line level: in many cases both printed and handwritten text occur on the same page. *There are sketches for how such branching could be implemented in HTRflow!*

Another direction is to use some kind of clustering to inspect the material before transcription. Our Norwegian and Danish colleagues always do this kind of analysis before transcribing a larger collection. That helps identify *outliers* that need special handling and can lead to a method that works well for the majority of the material. We do the reverse: we start with a method (our standard pipeline) and then choose archives that we think fit it well.
